# Tier 2 — DQN en LunarLander

Este notebook es una entrada independiente del Tier 1. Comparte los algoritmos de
`agents/` y la ejecución de `experiments/`; no contiene una copia alternativa del agente.

**VS Code:** abre la carpeta del proyecto, selecciona el kernel `.venv\Scripts\python.exe`
y usa **Shift+Enter** o **Ejecutar todo**. Tras actualizar el código, reinicia el kernel.
Las extensiones Python y Jupyter de Microsoft son necesarias. `Preparar_Proyecto.bat`
y `Preparar_Notebook.bat` instalan las dependencias del proyecto.

Los entrenamientos y la animación se activan expresamente en sus celdas. Ejecutar todo
consulta resultados guardados y ejemplos. Si un estudio sigue en curso, se indica como pendiente.

In [1]:
from pathlib import Path
import sys, json, copy
import numpy as np
import torch
from IPython.display import display, Markdown
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'agents/dqn.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Abre la carpeta completa del proyecto en VS Code.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.lunarlander import configuration, plan, make_env, train, evaluate
from experiments.ejecutar_tier23 import run
from experiments.cuaderno_lunar import OUTPUT, resumen, figura, discusion, demostrar
from experiments.cuaderno_tier1 import codigo
from agents.dqn import QNetwork, DQNAgent
from agents.replay import ReplayBuffer
from agents.double_dqn import DoubleDQNAgent
from agents.dqn_without_target import DQNWithoutTargetAgent
torch.set_num_threads(1)
torch.manual_seed(123)
CONFIG = configuration()
print('Python:', sys.executable)
print('Resultados:', OUTPUT)

Python: C:\Users\Harolt\Desktop\Universidad\Sem-02-2026\Agentes\Project 01\.venv\Scripts\python.exe
Resultados: C:\Users\Harolt\Desktop\Universidad\Sem-02-2026\Agentes\Project 01\results\tier23_lunarlander


## 1. Del carrito a una nave

En Tier 1 equilibrábamos una vara. Aquí intentamos aterrizar una nave en **LunarLander-v3**.
La observación contiene **8 valores**: posición horizontal/vertical, velocidades horizontal/vertical,
ángulo, velocidad angular y contacto de cada una de las dos patas con el suelo.
Son las observaciones del entorno, sin discretización tabular ni recompensa añadida.

Las **4 acciones** son: 0 no activar motores; 1 activar motor lateral izquierdo;
2 activar motor principal; 3 activar motor lateral derecho.

El objetivo del enunciado es **retorno medio ≥200 en 100 episodios**. Se premia acercarse,
reducir velocidad y aterrizar; usar motores tiene coste y estrellarse penaliza.

In [2]:
env = make_env(CONFIG)
try:
    obs, _ = env.reset(seed=123)
    print('Observación inicial:', obs)
    print('Observaciones:', env.observation_space)
    print('Acciones:', env.action_space)
    print('Máximo de pasos por episodio:', env.spec.max_episode_steps)
    print('Umbral declarado por la versión instalada:', env.spec.reward_threshold)
finally:
    env.close()

Observación inicial: [ 0.00517693  1.4033939   0.52433753 -0.3345171  -0.00599183 -0.11877016
  0.          0.        ]
Observaciones: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Acciones: Discrete(4)
Máximo de pasos por episodio: 1000
Umbral declarado por la versión instalada: 200


## 2. Parámetros y diseño del experimento

El diccionario es editable. Cambiarlo afecta entrenamientos nuevos, no los modelos ya guardados.
El presupuesto formal es `steps=500000`. El texto descriptivo `selection` del protocolo
congelado arrastra una errata que dice 200000; no gobierna el entrenamiento. Se conserva
el archivo histórico y se aclara aquí para mantener sus huellas de reproducción.
Usamos **tres semillas y el mismo presupuesto por modelo**. El baseline aparece en ambos
estudios y se reutiliza, evitando entrenar otra copia idéntica.

| Estudio | Qué cambia | Qué se mantiene |
|---|---|---|
| Tasa de aprendizaje | Tres valores de `learning_rate` | Los demás parámetros de base |
| Red objetivo | Tres valores de `target_sync_every` | Los demás parámetros de base |
| Ablación | Eliminar target y usar online en el objetivo | Replay, red, optimizador y presupuesto |

`hidden_widths`: neuronas; `batch_size`: experiencias por lote; `buffer_size`: memoria;
`learning_starts`: inicio de ajustes; `train_every`: pasos entre ajustes;
`gradient_steps`: ajustes realizados cada vez; `epsilon_decay_steps` y `epsilon_end`: exploración.

In [3]:
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
for nombre, trabajo in plan(CONFIG).items():
    print(nombre, '→', trabajo['role'], '| variante:', trabajo['variant'])

{
  "environment": "LunarLander-v3",
  "environment_kwargs": {
    "continuous": false,
    "enable_wind": false,
    "gravity": -10.0
  },
  "steps": 500000,
  "seeds": [
    42,
    43,
    44
  ],
  "eval_every": 50000,
  "periodic_eval_episodes": 10,
  "final_eval_episodes": 100,
  "threshold": 200.0,
  "evaluation_base": 10000000,
  "heldout_evaluation_base": 20000000,
  "base": {
    "hidden_widths": [
      128,
      128
    ],
    "learning_rate": 0.0005,
    "loss": "smooth_l1",
    "gamma": 0.99,
    "buffer_size": 100000,
    "batch_size": 64,
    "learning_starts": 5000,
    "train_every": 8,
    "gradient_steps": 1,
    "target_sync_every": 2000,
    "epsilon_decay_steps": 150000,
    "epsilon_end": 0.05
  },
  "studies": {
    "learning_rate": [
      0.0001,
      0.0005,
      0.001
    ],
    "target_sync_every": [
      500,
      2000,
      8000
    ]
  },
  "selection": "Modelos finales de 200000 pasos, sin escoger el mejor checkpoint. DQN base se reutiliza en amb

## 3. Reutilizamos nuestro DQN

La red ahora recibe 8 valores y devuelve 4 valores Q. Replay almacena transiciones de ocho
componentes. La red objetivo y la exploración cumplen la misma función que en Tier 1.
Estos bloques muestran **el código real** que ejecutan los experimentos.

In [4]:
codigo(QNetwork)
codigo(ReplayBuffer)

class QNetwork(nn.Module):
    """De observaciones continuas a un valor Q por acción discreta.

    Acepta un estado (obs_dim,) o un lote (batch_size, obs_dim).
    Devuelve (n_actions,) o (batch_size, n_actions), respectivamente.
    La salida es lineal: los valores Q no son probabilidades.
    """

    def __init__(self, obs_dim=4, n_actions=2, hidden_width=64):
        super().__init__()
        # Un entero mantiene dos capas iguales; una pareja permite, por ejemplo, 120/84.
        widths = (hidden_width, hidden_width) if isinstance(hidden_width, int) else tuple(hidden_width)
        if len(widths) != 2 or min(obs_dim, n_actions, *widths) < 1:
            raise ValueError("Las dimensiones deben ser positivas")
        self.layers = nn.Sequential(
            nn.Linear(obs_dim, widths[0]),
            nn.ReLU(),
            nn.Linear(widths[0], widths[1]),
            nn.ReLU(),
            nn.Linear(widths[1], n_actions),
        )

    def forward(self, observations):
        return self.layers(observations)

class ReplayBuffer:
    """Guardar transiciones y muestrearlas uniformemente sin reemplazo.

    Al llenarse, cada experiencia nueva sustituye la más antigua. Las dos
    observaciones se copian para evitar cambios al reutilizar arrays del entorno.
    terminated y truncated se conservan separados para calcular objetivos luego.
    """

    def __init__(self, capacity, observation_shape=(4,), seed=42):
        if not isinstance(capacity, int) or isinstance(capacity, bool) or capacity < 1:
            raise ValueError("capacity debe ser un entero positivo")
        self.capacity = capacity
        self.observation_shape = tuple(observation_shape)
        self.observations = np.empty((capacity,) + self.observation_shape, dtype=np.float32)
        self.next_observations = np.empty_like(self.observations)
        self.actions = np.empty(capacity, dtype=np.int64)
        self.rewards = np.empty(capacity, dtype=np.float32)
        self.terminated = np.empty(capacity, dtype=bool)
        self.truncated = np.empty(capacity, dtype=bool)
        self._position = 0
        self._size = 0
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self._size

    def add(self, observation, action, reward, next_observation, terminated, truncated):
        observation = np.asarray(observation, dtype=np.float32)
        next_observation = np.asarray(next_observation, dtype=np.float32)
        if (observation.shape != self.observation_shape or
                next_observation.shape != self.observation_shape):
            raise ValueError("Las observaciones no coinciden con la forma de la memoria")
        index = self._position
        self.observations[index] = observation
        self.actions[index] = action
        self.rewards[index] = reward
        self.next_observations[index] = next_observation
        self.terminated[index] = terminated
        self.truncated[index] = truncated
        self._position = (index + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size):
        if (not isinstance(batch_size, int) or isinstance(batch_size, bool) or
                not 1 <= batch_size <= self._size):
            raise ValueError("batch_size debe estar entre 1 y la cantidad guardada")
        indices = self.rng.choice(self._size, size=batch_size, replace=False)
        # El indexado con un array produce copias: el lote no modifica la memoria.
        return ReplayBatch(
            self.observations[indices], self.actions[indices], self.rewards[indices],
            self.next_observations[indices], self.terminated[indices], self.truncated[indices],
        )

In [5]:
codigo(DQNAgent)

class DQNAgent:
    """La red online aprende; target conserva una referencia entre copias.

    El bucle de entrenamiento decidirá cuándo llamar a sync_target(), medido
    en pasos del entorno. learn() no sincroniza la referencia automáticamente.
    """

    def __init__(self, obs_dim=4, n_actions=2, hidden_width=64,
                 learning_rate=1e-3, gamma=0.99, seed=42, loss="smooth_l1"):
        if learning_rate <= 0 or not 0 <= gamma <= 1:
            raise ValueError("learning_rate debe ser positiva y gamma estar en [0, 1]")
        if loss not in ("smooth_l1", "mse"):
            raise ValueError("loss debe ser smooth_l1 o mse")
        torch.manual_seed(seed)
        self.online = QNetwork(obs_dim, n_actions, hidden_width)
        self.target = deepcopy(self.online)
        self.target.requires_grad_(False)
        self.target.eval()
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=learning_rate)
        self.gamma = gamma
        self.n_actions = n_actions
        self.loss_name = loss
        self.rng = np.random.default_rng(seed)

    def choose_action(self, observation, epsilon):
        if not 0 <= epsilon <= 1:
            raise ValueError("epsilon debe estar en [0, 1]")
        if self.rng.random() < epsilon:
            return int(self.rng.integers(self.n_actions))
        with torch.no_grad():
            values = self.online(torch.as_tensor(observation, dtype=torch.float32)).numpy()
        best_actions = np.flatnonzero(values == values.max())
        return int(self.rng.choice(best_actions))

    def learn(self, batch):
        observations = torch.as_tensor(batch.observations, dtype=torch.float32)
        actions = torch.as_tensor(batch.actions, dtype=torch.int64)
        rewards = torch.as_tensor(batch.rewards, dtype=torch.float32)
        next_observations = torch.as_tensor(batch.next_observations, dtype=torch.float32)
        terminated = torch.as_tensor(batch.terminated, dtype=torch.bool)

        # De las dos salidas solo comparamos la correspondiente a la acción tomada.
        predictions = self.online(observations).gather(1, actions[:, None]).squeeze(1)
        with torch.no_grad():
            next_values = self.bootstrap_values(next_observations)
            # Un corte por tiempo no elimina el valor futuro: solo terminated.
            targets = rewards + self.gamma * (~terminated).float() * next_values

        if self.loss_name == "mse":
            loss = nn.functional.mse_loss(predictions, targets)
        else:
            loss = nn.functional.smooth_l1_loss(predictions, targets)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), max_norm=10.0)
        self.optimizer.step()
        return {"loss": float(loss.detach()),
                "predictions_before": predictions.detach().tolist(),
                "targets": targets.tolist()}

    def sync_target(self):
        """Actualizar la referencia mediante una copia completa de la red online."""
        self.target.load_state_dict(self.online.state_dict())

    def bootstrap_values(self, next_observations):
        """DQN estándar: seleccionar y valorar con la red objetivo."""
        return self.target(next_observations).max(dim=1).values

## 4. Cómo se construye el objetivo

\[
y=r+\gamma(1-\mathrm{terminated})\max_a Q_{target}(s',a)
\]

El fallo/fin real elimina el término futuro. Un corte por tiempo no lo elimina.
El valor Q no es una probabilidad. Tampoco usamos el límite Q≤100 de CartPole:
las recompensas de LunarLander son diferentes.

In [6]:
reward, gamma, next_best_q = 1.0, 0.9, 7.0
for terminated, truncated in [(False, False), (True, False), (False, True)]:
    target = reward + gamma * (not terminated) * next_best_q
    print('terminated:', terminated, '| truncated:', truncated, '→ objetivo:', target)

terminated: False | truncated: False → objetivo: 7.3
terminated: True | truncated: False → objetivo: 1.0
terminated: False | truncated: True → objetivo: 7.3


## 5. Ablación: ¿para qué sirve target?

La ablación conserva replay y todos los parámetros. Elimina la red independiente y calcula
el bootstrap con **online**, sin propagar gradiente por ese objetivo. Si empeora, aporta
evidencia de estabilización en esta configuración; no asumimos el resultado antes de medirlo.

In [7]:
codigo(DQNWithoutTargetAgent)

class DQNWithoutTargetAgent(DQNAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        del self.target

    def bootstrap_values(self, next_observations):
        return self.online(next_observations).max(dim=1).values

    def sync_target(self):
        """No hay una red objetivo que actualizar en esta ablación."""

## 6. Entrenamiento y evaluación separados

Cada resultado formal incluye tres semillas. Cada cierto número de pasos evaluamos 10
episodios greedy; al inicio/final, 100. Confirmamos con 100 cuando un control supera 200.
Se guarda el modelo final, aunque haya tenido un control anterior mejor.

Evaluar usa otro entorno, una copia de la red y un generador independiente. Entrenamiento,
pérdidas y evaluaciones se guardan por separado en CSV.

In [8]:
codigo(evaluate)

def evaluate(network, config, seed, episodes, evaluation_base=None):
    """Otro entorno y RNG; no consumir estados aleatorios del entrenamiento."""
    evaluator = deepcopy(network).eval()
    base = config["evaluation_base"] if evaluation_base is None else evaluation_base
    rng = np.random.default_rng(base + 500_000 + 1000 * seed)
    env = make_env(config)
    returns = []
    try:
        for episode in range(episodes):
            obs, _ = env.reset(seed=base + 1000 * seed + episode)
            total = 0.0
            while True:
                with torch.inference_mode():
                    scores = evaluator(torch.as_tensor(obs, dtype=torch.float32)).numpy()
                action = int(rng.choice(np.flatnonzero(scores == scores.max())))
                obs, reward, terminated, truncated, _ = env.step(action)
                total += reward
                if terminated or truncated:
                    break
            returns.append(float(total))
    finally:
        env.close()
    return returns

In [9]:
codigo(train)

def train(config, job, seed, folder):
    """Mismo bucle para todos los tratamientos; solo cambian parámetros o agente."""
    folder = Path(folder)
    p = job["parameters"]
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    folder.mkdir(parents=True, exist_ok=False)
    env = make_env(config)
    obs_dim, actions = int(env.observation_space.shape[0]), int(env.action_space.n)
    agent = CLASSES[job["variant"]](obs_dim=obs_dim, n_actions=actions,
            hidden_width=p["hidden_widths"], learning_rate=p["learning_rate"], gamma=p["gamma"],
            seed=seed, loss=p["loss"])
    replay = ReplayBuffer(p["buffer_size"], observation_shape=(obs_dim,), seed=4_000_000 + seed)
    probes = probe_states(config, seed)
    source_paths = ["agents/dqn.py", "agents/double_dqn.py", "agents/dqn_without_target.py", "agents/replay.py", "experiments/lunarlander.py"]
    run_config = {"input_hash": fingerprint(config, job, seed), "protocol": config, "job": job, "seed": seed,
        "architecture": [obs_dim, *p["hidden_widths"], actions], "environment_max_steps": env.spec.max_episode_steps,
        "environment_declared_threshold": env.spec.reward_threshold,
        "python": platform.python_version(), "platform": platform.platform(), "processor": platform.processor(),
        "logical_cpus": os.cpu_count(), "device": "cpu", "torch_threads": 1,
        "packages": {name: metadata.version(name) for name in ("gymnasium", "numpy", "torch", "Box2D")},
        "source_hashes": {path: hashlib.sha256((ROOT / path).read_bytes()).hexdigest() for path in source_paths},
        "evaluation_reset_seed": "evaluation_base + 1000 * training_seed + episode",
        "evaluation_action_seed": "evaluation_base + 500000 + 1000 * training_seed",
        "replay_seed": 4_000_000 + seed,
        "target_network": job["variant"] != "no_target"}
    (folder / "config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
    eval_rows, confirmations = [], []
    training_seconds = 0.0
    updates = syncs = 0
    started = time.perf_counter()

    def checkpoint(step, episodes):
        returns = evaluate(agent.online, config, seed, episodes)
        mean = float(np.mean(returns))
        with torch.inference_mode():
            online_q = agent.online(probes)
            if hasattr(agent, "target"):
                target_q = agent.target(probes)
                choices = online_q.argmax(1, keepdim=True)
                gap = float((target_q.max(1).values - target_q.gather(1, choices).squeeze(1)).mean())
            else:
                gap = None
        eval_rows.append({"training_steps": step, "episodes": episodes, "mean_return": mean,
            "std_return": float(np.std(returns)), "returns": json.dumps(returns),
            "training_seconds": training_seconds, "total_seconds": time.perf_counter() - started,
            "q_probe_min": float(online_q.min()), "q_probe_max": float(online_q.max()),
            "target_selection_gap": gap})
        if episodes == config["final_eval_episodes"] or mean >= config["threshold"]:
            confirmed = returns if episodes == config["final_eval_episodes"] else evaluate(agent.online, config, seed, config["final_eval_episodes"])
            confirmations.append({"training_steps": step, "mean_return": float(np.mean(confirmed)),
                "threshold_met": bool(np.mean(confirmed) >= config["threshold"]), "returns": confirmed})
        with (folder / "evaluation.csv").open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=list(eval_rows[0]))
            writer.writeheader()
            writer.writerows(eval_rows)
        (folder / "threshold_checks.json").write_text(json.dumps(confirmations, indent=2), encoding="utf-8")
        print(f"{folder.parent.name} | semilla {seed} | paso {step:>6} | evaluación {episodes}: {mean:.2f}", flush=True)

    step = 0
    try:
        checkpoint(0, config["final_eval_episodes"])
        obs, _

## Ejecutar una prueba corta (opcional)

La prueba usa un modelo pequeño y una semilla para comprobar el recorrido. **No es un
resultado formal** y no pretende alcanzar el objetivo. Se guarda en una carpeta nueva.

In [10]:
EJECUTAR_PRUEBA = False
if EJECUTAR_PRUEBA:
    from datetime import datetime
    prueba = copy.deepcopy(CONFIG)
    prueba.update(steps=2000, eval_every=2000, seeds=[123])
    prueba['base'].update(hidden_widths=[32,32], buffer_size=2000, batch_size=32,
                         learning_starts=128, gradient_steps=1)
    destino = ROOT / 'results/notebook_lunar' / datetime.now().strftime('prueba_%Y%m%d_%H%M%S_%f')
    run(prueba, destino, groups=['base'])
else:
    print('Prueba corta desactivada.')

Prueba corta desactivada.


## Repetir Tier 2 (opcional, entrenamiento largo)

Activa la celda solo para volver a entrenar. La carpeta debe ser nueva si cambias parámetros.
Para continuar el protocolo original puedes usar OUTPUT: se reutilizan resultados completos
con el mismo protocolo y se rechaza sobrescribir ejecuciones parciales.

Los estudios formales necesitan al menos tres semillas. Tier 2 comprende seis grupos únicos;
Tier 3 comprende base y Double DQN. Si base ya existe, puede reutilizarse.

In [11]:
EJECUTAR_TIER2 = False
if EJECUTAR_TIER2:
    from datetime import datetime
    if len(set(CONFIG['seeds'])) < 3:
        raise ValueError('El experimento formal requiere tres semillas independientes.')
    destino = ROOT / 'results/notebook_lunar' / datetime.now().strftime('tier2_%Y%m%d_%H%M%S_%f')
    # Para continuar el mismo protocolo original: destino = OUTPUT
    run(CONFIG, destino, groups=[n for n in plan(CONFIG) if n != 'double'])
else:
    print('Entrenamiento formal desactivado. Los resultados guardados se consultan abajo.')

Entrenamiento formal desactivado. Los resultados guardados se consultan abajo.


## Resultados guardados

Las tablas usan 100 episodios por modelo. La DE describe variación entre las medias de
las tres semillas. Las curvas usan 10 episodios por control, incluidos los primeros 10
al inicio/final; por ello su último punto puede diferir de la tabla final de 100.
Editar CONFIG no cambia estos resultados: corresponden al protocolo guardado en OUTPUT.

In [12]:
filas = resumen(tier=2)

Tier 2: 0/6 grupos completos; cada uno debe incluir tres semillas.
Todavía no hay tres semillas completas para mostrar resultados finales.


### Estudio de tasa de aprendizaje: solo cambia learning_rate

In [13]:
figura('learning_rate')

Figura pendiente: learning_rate. Se genera al completar las tres semillas de cada tratamiento.


### Estudio de target: solo cambia target_sync_every

In [14]:
figura('target_sync_every')

Figura pendiente: target_sync_every. Se genera al completar las tres semillas de cada tratamiento.


### Ablación: DQN base frente a DQN sin red objetivo

In [15]:
figura('ablation')

Figura pendiente: ablation. Se genera al completar las tres semillas de cada tratamiento.


In [16]:
discusion(tier=2)

# Tier 2 — LunarLander-v3

## Protocolo

Entorno LunarLander-v3, acciones discretas, parámetros {'continuous': False, 'enable_wind': False, 'gravity': -10.0}.
Presupuesto 500000 pasos, semillas [42, 43, 44], evaluación greedy cada 50000 pasos.
Controles de 10 episodios; inicio/final de 100 y confirmación de 100 cuando un control alcanza 200.
Se entregan modelos finales. No se elige el mejor checkpoint ni se descarta una semilla débil.
Las curvas usan los primeros 10 episodios al inicio/final; las tablas finales usan los 100.
La banda es una DE entre las medias de las semillas (ddof=0), no un IC del 95%.
DQN base se reutiliza en los dos estudios, la ablación y Tier 3. No son entrenamientos adicionales independientes.
Todos los agentes son propios; la referencia SB3 Zoo solo orientó parámetros. No se importó un agente entrenado.

## Estado y resultados completos

Grupos completos: 0/6. No se informan ejecuciones parciales como resultados finales.

| Grupo | Media final ± DE | Retornos por semilla | Semillas ≥200 | Entrenamiento medio |
|---|---:|---|---:|---:|

Tiempos excluyen evaluación, dependen de esta máquina y no son benchmarks universales.
Los experimentos formales se ejecutan secuencialmente en CPU con un hilo PyTorch.
Durante el trabajo se realizan verificaciones y mediciones de coste; los tiempos no son benchmarks aislados.

## Hiperparámetros


## Diagnóstico y límites

LunarLander tiene recompensas distintas de CartPole. No se usa el límite Q≤100 del análisis anterior.
Los CSV registran pérdidas, extremos Q en 256 estados y una brecha de selección entre online y target.
Esa brecha no es el error Q frente al valor verdadero ni demuestra sobreestimación por sí sola.
Las semillas se usan durante el desarrollo; tres repeticiones son el mínimo, no una garantía universal.
La comparación causal de hiperparámetros depende de mantener el resto del protocolo; revisar config.json por modelo.
Tier 2 requiere ambos estudios (dos parámetros, tres valores cada uno) y la ablación; Tier 3 se documenta por separado.
La brecha entre redes se registra después de las copias programadas y puede ser cero por esa coincidencia temporal.

## Pasos hasta el umbral


## Referencias

- Gymnasium LunarLander: https://gymnasium.farama.org/environments/box2d/lunar_lander/
- Referencia de parámetros: https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml
- Double DQN: van Hasselt, Guez y Silver, https://arxiv.org/abs/1509.06461


## Ver la nave dentro del notebook

Se muestra el primer episodio de la semilla escogida. No aprende durante la animación.
La recompensa de un episodio no sustituye la evaluación de 100 ni la repetición de semillas.

In [17]:
VER_NAVE = False
GRUPO = 'base'
SEMILLA = 42  # 42, 43 o 44
if VER_NAVE:
    demostrar(GRUPO, SEMILLA, inline=True)
else:
    print('Cambia VER_NAVE a True para generar la animación.')

Cambia VER_NAVE a True para generar la animación.

## Preguntas para la defensa

- ¿Qué observan las ocho entradas y qué representan las cuatro salidas Q?
- ¿Cómo distinguimos entrenar de evaluar? ¿Por qué tres semillas y 100 episodios finales?
- ¿Qué cambia exactamente en cada comparación y qué permanece igual?
- ¿Qué evidencia dan las curvas, la variación y los resultados negativos?
- ¿Por qué un Q grande o una brecha entre redes no prueba por sí sola sobreestimación verdadera?

El artículo final debe incorporar los resultados e interpretación. Los notebooks no sustituyen
el paper ni la defensa. Uso de IA: `report/uso_ia.md`.

Referencias: [Gymnasium](https://gymnasium.farama.org/environments/box2d/lunar_lander/),
[parámetros de referencia](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml),
[Double DQN](https://arxiv.org/abs/1509.06461).